> **Public archive note.** Archived source from a previously executed Kaggle session. Cell outputs removed for privacy and archive size. Execute from start with the checksum-matched frozen dataset and preserved model checkpoints; full fresh-run reproducibility not verified. Related text-free results are in results/external_turkey_syria_2023/.


# IJIES Round 2 — Independent Turkey–Syria 2023 frozen evaluation

**Use Kaggle**. This notebook evaluates a real archived, fixed-split **full VIRCON-ERT seed 20266008** checkpoint against a new 2023 event and a matched TF-IDF + LinearSVC baseline. **It does not retrain VIRCON-ERT** or manufacture labels, checkpoints or predictions. The historical 120-run event folds remain development evidence, not independent confirmation.

### Kaggle setup (Add Input)

1. `IJIES_Independent_Event_Adjudicated_600.csv` — master ground truth (600 rows).
2. `IJIES_Five_Label_Confirm_Template.csv` **after independent human review**, uploaded under the name `IJIES_Five_Label_Confirmed.csv`. The reviewer must fill `review_status=confirmed` and a nonblank `review_notes` for each of the five rows, and correct any final label as needed. Do **not** assume AI suggestions are human review.
3. `IJIES_Round2_Experiments-20261116T124209Z-1-001.zip` — original experiments ZIP (~235 MB) containing `fixed_full_seed20266008/model_state_dict.pt`, tokenizer, `model_structure.json`, and `run_manifest.json`. If you have already attached the extracted `fixed_full_seed20266008/` folder as a Kaggle dataset, that also works.
4. The original `crisis_consolidated_humanitarian_filtered_lang_en_train.tsv` (dataset `gomathybalaji/crisis`) to train the sparse baseline on the **same historical training partition**.

Set accelerator to **T4 GPU** (CPU works but is slower), internet is optional if `transformers` is already installed. Execute **Cells 1–6 in order**. The final cell exports results and a ZIP; no source tweet text is included in the ZIP.

**Interpretation limits:** One new disaster event; rare independent request positives; no event-cluster confidence interval possible. A reviewed label is not evidence of two human raters unless two real people performed the annotations. Model selection and threshold optimization must not use this evaluation dataset. The crisis-domain encoder was stronger on the 120-run exploratory grid and is **not** included here without an actual trained crisis-domain checkpoint.

In [ ]:
# CELL 1 — Preflight inputs and explain the five confirmation decisions.
from pathlib import Path
import hashlib, json, os, re, csv, shutil, zipfile, sys, subprocess, importlib.util
import pandas as pd
import numpy as np

WORK = Path('/kaggle/working/IJIES_Independent_Frozen_Eval')
WORK.mkdir(parents=True, exist_ok=True)
INPUTS = [Path('/kaggle/input'), Path('/kaggle/working')]

def locate_exact(filename, required=True):
    paths=[]
    for root in INPUTS:
        paths.extend(p for p in root.rglob(filename) if p.is_file())
    paths=sorted(set(paths), key=lambda p:str(p))
    if not paths:
        if required: raise FileNotFoundError(f'Attach {filename} via Kaggle Add Input, then rerun.')
        return None
    # Multiple copies are acceptable only if byte-identical.
    if len(paths)>1:
        sizes={p.stat().st_size for p in paths}
        if len(sizes)>1 or len({hashlib.sha256(p.read_bytes()).hexdigest() for p in paths})>1:
            raise RuntimeError(f'Multiple different files named {filename}: {paths}')
    return paths[0]

master_path=locate_exact('IJIES_Independent_Event_Adjudicated_600.csv')
confirmed_path=locate_exact('IJIES_Five_Label_Confirmed.csv',required=False)
print('Master file:',master_path)
print('Confirmed five-label file:',confirmed_path or 'MISSING — complete human review first')
print('Output folder:',WORK)

if confirmed_path is None:
    print("""
ACTION REQUIRED BEFORE CELL 2:
- Open IJIES_Five_Label_Confirm_Template.csv in Excel.
- Review all five source messages and their current labels.
- Set review_status to confirmed in all five rows.
- Enter a meaningful review_notes explanation for each.
- Make any evidence-supported corrections in confirmed_* columns.
- Save as IJIES_Five_Label_Confirmed.csv and attach to Kaggle.
No ground-truth label should be silently changed by code.
""")

In [ ]:
# CELL 2 — Apply five confirmed reviews and FREEZE dataset before inference.
from sklearn.metrics import cohen_kappa_score

confirm_path=locate_exact('IJIES_Five_Label_Confirmed.csv')
master=pd.read_csv(master_path,dtype=str).fillna('')
reviews=pd.read_csv(confirm_path,dtype=str).fillna('')
for df in (master,reviews):
    df.columns=df.columns.str.strip()
    for col in df.columns:
        df[col]=df[col].astype(str).str.strip()

LABELS=['affected_individual','caution_and_advice','displaced_and_evacuations',
        'donation_and_volunteering','infrastructure_and_utilities_damage',
        'injured_or_dead_people','missing_and_found_people','not_humanitarian',
        'requests_or_needs','response_efforts','sympathy_and_support']
TRIAGE=['other','request','response']
VALID_A=set(TRIAGE)|{'uncertain'}

assert len(master)==600 and master.message_id.is_unique
assert len(reviews)==5 and reviews.message_id.is_unique
assert set(['message_id','event_id','text','disaster_relevance','humanitarian_label_11','actionability_label']).issubset(master.columns)
need={'message_id','confirmed_humanitarian_label_11','confirmed_actionability_label','review_status','review_notes'}
if missing:=need-set(reviews.columns): raise ValueError(f'Confirmed-review columns missing: {sorted(missing)}')
if not set(reviews.message_id).issubset(set(master.message_id)):
    raise ValueError('Review IDs not found in the master.')
if not reviews.review_status.str.casefold().eq('confirmed').all():
    raise ValueError('All five review_status values must be confirmed by human reviewer.')
if not reviews.review_notes.str.len().ge(8).all():
    raise ValueError('Each review_notes needs a meaningful explanation (>=8 characters).')
if not reviews.confirmed_humanitarian_label_11.isin(LABELS).all():
    raise ValueError('Invalid confirmed humanitarian label in the five-row file.')
if not reviews.confirmed_actionability_label.isin(VALID_A).all():
    raise ValueError('Invalid confirmed actionability label in the five-row file.')

expected_file=locate_exact('IJIES_Final_Label_Review_Flags_5.csv',required=False)
if expected_file is not None:
    expected_ids=set(pd.read_csv(expected_file,dtype=str).message_id)
    if set(reviews.message_id)!=expected_ids: raise ValueError('Confirmed file IDs differ from the original five flags.')
else:
    print('NOTE: Original flags CSV not attached; checking all five reviews against master only.')

frozen=master.copy().set_index('message_id',verify_integrity=True)
changes=[]
for row in reviews.itertuples(index=False):
    mid=row.message_id
    before11=frozen.at[mid,'humanitarian_label_11']
    before3=frozen.at[mid,'actionability_label']
    frozen.at[mid,'humanitarian_label_11']=row.confirmed_humanitarian_label_11
    frozen.at[mid,'actionability_label']=row.confirmed_actionability_label
    changes.append({'message_id':mid,'old_humanitarian':before11,
                    'new_humanitarian':row.confirmed_humanitarian_label_11,
                    'old_actionability':before3,'new_actionability':row.confirmed_actionability_label,
                    'review_status':'confirmed','review_notes':row.review_notes})
frozen=frozen.reset_index().sort_values('message_id').reset_index(drop=True)
assert len(frozen)==600 and frozen.message_id.is_unique
assert frozen.disaster_relevance.isin({'relevant','not_relevant','uncertain'}).all()
assert frozen.humanitarian_label_11.isin(LABELS).all()
assert frozen.actionability_label.isin(VALID_A).all()
assert frozen.event_id.nunique()==1
assert frozen.event_id.iloc[0]=='turkey_syria_earthquake_2023'
assert frozen.text.ne('').all()

frozen_path=WORK/'IJIES_Independent_Event_FROZEN_600.csv'
if frozen_path.exists():
    old_hash=hashlib.sha256(frozen_path.read_bytes()).hexdigest()
    new_payload=frozen.to_csv(index=False).encode('utf-8')
    if hashlib.sha256(new_payload).hexdigest()!=old_hash:
        raise RuntimeError('Different frozen labels already exist. Do not overwrite after evaluation; start a new version with documented rationale.')
else:
    frozen.to_csv(frozen_path,index=False)
label_hash=hashlib.sha256(frozen_path.read_bytes()).hexdigest()
review_df=pd.DataFrame(changes)
review_df.to_csv(WORK/'Five_Human_Review_Decisions.csv',index=False)
metadata={'dataset':'2023 Turkey-Syria earthquake', 'n':int(len(frozen)),
    'events':int(frozen.event_id.nunique()),'frozen_dataset_sha256':label_hash,
    'human_label_review':'Five rows confirmed by external reviewer, per supplied file',
    'notes':'Do not use these labels for training, checkpoint selection, or threshold tuning.'}
(WORK/'Frozen_Dataset_Manifest.json').write_text(json.dumps(metadata,indent=2))
print('FROZEN: 600 rows; SHA256:',label_hash)
print('Relevant rows:',sum(frozen.disaster_relevance=='relevant'))
print('Primary triage rows (relevant, non-uncertain):',
      sum((frozen.disaster_relevance=='relevant')&frozen.actionability_label.isin(TRIAGE)))
print('Confirmed label changes:',sum(review_df.old_actionability!=review_df.new_actionability)+
    sum(review_df.old_humanitarian!=review_df.new_humanitarian))
print('Actionability counts:',frozen.actionability_label.value_counts().to_dict())

In [ ]:
# CELL 3 — Load the GENUINE archived checkpoint and saved tokenizer (no training).
import torch, torch.nn as nn

if importlib.util.find_spec('transformers') is None:
    print('Installing transformers (Internet access required for this step).')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5'])
from transformers import AutoTokenizer, DistilBertModel, DistilBertConfig

# Locate an already extracted model folder first.
model_candidates=[]
for root in INPUTS:
    for p in root.rglob('model_structure.json'):
        if (p.parent.name=='fixed_full_seed20266008' and
            (p.parent/'model_state_dict.pt').exists() and
            (p.parent/'tokenizer').exists()):
            model_candidates.append(p.parent)

if model_candidates:
    model_dir=sorted(set(model_candidates),key=str)[0]
    print('Existing checkpoint directory:',model_dir)
else:
    # Use original IJIES experiments ZIP. No fabricated checkpoint accepted.
    archives=[]
    for root in INPUTS:
        archives += list(root.rglob('*.zip'))
    prefix='IJIES_Round2_Experiments/fixed_full_seed20266008/'
    chosen=None
    for zp in sorted(set(archives),key=str):
        try:
            with zipfile.ZipFile(zp) as z:
                names=set(z.namelist())
                if {prefix+'model_state_dict.pt',prefix+'model_structure.json',
                    prefix+'run_manifest.json',prefix+'tokenizer/tokenizer.json'}.issubset(names):
                    chosen=zp
                    break
        except zipfile.BadZipFile:
            continue
    if chosen is None:
        raise FileNotFoundError(
            'Attach IJIES_Round2_Experiments-20261116T124209Z-1-001.zip '
            'or a Kaggle dataset containing its fixed_full_seed20266008/ checkpoint folder.')
    model_dir=WORK/'fixed_full_seed20266008'
    model_dir.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(chosen) as z:
        for member in z.namelist():
            if not member.startswith(prefix) or member.endswith('/'):
                continue
            rel=Path(member[len(prefix):])
            if '..' in rel.parts or rel.is_absolute():
                raise RuntimeError('Unsafe path within original checkpoint archive.')
            dest=model_dir/rel
            dest.parent.mkdir(parents=True,exist_ok=True)
            if not dest.exists():
                with z.open(member) as a,dest.open('wb') as b:
                    shutil.copyfileobj(a,b)
    print('Extracted real checkpoint from:',chosen)

meta=json.loads((model_dir/'model_structure.json').read_text())
manifest=json.loads((model_dir/'run_manifest.json').read_text())
assert manifest['protocol']=='fixed' and manifest['variant']=='full' and int(manifest['seed'])==20266008
assert int(manifest['n_train'])==61164 and not manifest['debug_subset']
assert meta['labels']==LABELS
assert meta['encoder']=='distilbert-base-uncased'
assert meta['inference_request_override'] is False
assert int(meta['input_length'])==128
assert meta['triage']==TRIAGE

# Matches archived scripts/model.py parameter names, including the event discriminator.
class ArchivedFullVIRCON(nn.Module):
    def __init__(self,n_labels,n_events):
        super().__init__()
        self.encoder=DistilBertModel(DistilBertConfig())
        self.dropout=nn.Dropout(0.20)
        h=self.encoder.config.hidden_size
        self.humanitarian=nn.Linear(h,n_labels)
        self.triage=nn.Linear(h,3)
        self.request=nn.Linear(h,1)
        self.event_discriminator=nn.Sequential(
            nn.Linear(h,128),nn.ReLU(),nn.Dropout(0.10),nn.Linear(128,n_events))
    def forward(self,ids,mask):
        hidden=self.encoder(input_ids=ids,attention_mask=mask).last_hidden_state[:,0]
        dropped=self.dropout(hidden)
        return (self.humanitarian(dropped),self.triage(dropped),
                self.request(dropped).squeeze(-1))

checkpoint_path=model_dir/'model_state_dict.pt'
ckpt_hash=hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=ArchivedFullVIRCON(len(LABELS),int(meta['source_event_classes']))
state=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
model.load_state_dict(state,strict=True)
del state
model=model.to(device).eval()
tokenizer=AutoTokenizer.from_pretrained(str(model_dir/'tokenizer'),local_files_only=True)
print('Checkpoint SHA256:',ckpt_hash)
print('Model:',manifest['variant'],'seed:',manifest['seed'],'n_train:',manifest['n_train'])
print('Device:',device,'| TRUE frozen checkpoint loaded strictly.')

In [ ]:
# CELL 4 — Real VIRCON-ERT frozen inference (NO label access for prediction).
import html
import torch.nn.functional as F

URL_RE=re.compile(r'(?:https?://\S+|www\.\S+)',re.I)
USER_RE=re.compile(r'@[A-Za-z0-9_]+')
HASHTAG_RE=re.compile(r'#([A-Za-z0-9_]+)')
NUMBER_RE=re.compile(r'\b\d+(?:[.,]\d+)?\b')
NON_TOKEN_RE=re.compile(r"[^a-z0-9_\s']+")
SPACE_RE=re.compile(r'\s+')
def normalise_text(value):
    if pd.isna(value):return ''
    value=html.unescape(str(value)).lower()
    value=URL_RE.sub(' urltoken ',value)
    value=USER_RE.sub(' usertoken ',value)
    value=HASHTAG_RE.sub(r' \1 ',value)
    value=NUMBER_RE.sub(' numtoken ',value)
    return SPACE_RE.sub(' ',NON_TOKEN_RE.sub(' ',value)).strip()

text_input=frozen.text.map(normalise_text).tolist()
assert all(text_input),'Empty text after training-time normalization.'
probs11=[];probs3=[];request_probs=[]
BATCH=32 if device.type=='cuda' else 8
with torch.inference_mode():
    for start in range(0,len(text_input),BATCH):
        texts=text_input[start:start+BATCH]
        enc=tokenizer(texts,padding='max_length',truncation=True,
                      max_length=128,return_tensors='pt')
        z11,z3,zreq=model(enc['input_ids'].to(device),enc['attention_mask'].to(device))
        probs11.append(F.softmax(z11.float(),dim=-1).cpu().numpy())
        probs3.append(F.softmax(z3.float(),dim=-1).cpu().numpy())
        request_probs.append(torch.sigmoid(zreq.float()).cpu().numpy())
P11=np.vstack(probs11);P3=np.vstack(probs3);PR=np.concatenate(request_probs)
assert P11.shape==(600,11) and P3.shape==(600,3) and PR.shape==(600,)
assert np.isfinite(P11).all() and np.isfinite(P3).all() and np.isfinite(PR).all()

full_pred=pd.DataFrame({'message_id':frozen.message_id,
    'event_id':frozen.event_id,
    'pred_11':[LABELS[int(i)] for i in P11.argmax(axis=1)],
    'pred_triage':[TRIAGE[int(i)] for i in P3.argmax(axis=1)],
    'request_probability':PR})
for j,lbl in enumerate(LABELS): full_pred['prob_11_'+lbl]=P11[:,j]
for j,lbl in enumerate(TRIAGE): full_pred['prob_triage_'+lbl]=P3[:,j]
full_pred_path=WORK/'VIRCON_Full_20266008_External_Predictions_No_Text.csv'
full_pred.to_csv(full_pred_path,index=False)
print('TRUE VIRCON predictions:',len(full_pred),'| no hard override')
print('Saved:',full_pred_path)
print('Predicted humanitarian class supports:',full_pred.pred_11.value_counts().to_dict())
print('Predicted direct triage supports:',full_pred.pred_triage.value_counts().to_dict())

In [ ]:
# CELL 5 — Matched TF-IDF + LinearSVC trained on original CrisisBench TRAIN ONLY.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

train_file=locate_exact('crisis_consolidated_humanitarian_filtered_lang_en_train.tsv')
train_sha=hashlib.sha256(train_file.read_bytes()).hexdigest()
if train_sha!=manifest['train_sha256']:
    raise RuntimeError('Historical train TSV differs from fixed checkpoint training data: SHA256 mismatch.')
orig=pd.read_csv(train_file,sep='\t',low_memory=False)
need={'text','class_label','event'}
assert need.issubset(orig.columns),f'Unexpected TSV columns: {orig.columns.tolist()}'
orig['clean_text']=orig['text'].map(normalise_text)
orig['class_label']=orig.class_label.fillna('').astype(str).str.strip()
train=orig.loc[orig.clean_text.ne('') & orig.class_label.ne('')].reset_index(drop=True)
assert len(train)==61164,f'Expected 61164 training records, found {len(train)}.'
assert train.class_label.isin(LABELS).all()
print('Training sparse baseline on original',len(train),'samples; NO new-event labels used.')

vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=0.95,max_features=50000)
X_train=vectorizer.fit_transform(train.clean_text.tolist())
clf=LinearSVC(C=1.0,class_weight='balanced',random_state=20266008)
clf.fit(X_train,train.class_label)
X_ext=vectorizer.transform(text_input)
spred=clf.predict(X_ext)

def triage_from_humanitarian(label):
    if label=='requests_or_needs':return 'request'
    if label in ('response_efforts','donation_and_volunteering'):return 'response'
    return 'other'

sparse_pred=pd.DataFrame({'message_id':frozen.message_id,
    'event_id':frozen.event_id,
    'pred_11':spred,
    'pred_triage':[triage_from_humanitarian(x) for x in spred]})
sparse_path=WORK/'TFIDF_SVC_External_Predictions_No_Text.csv'
sparse_pred.to_csv(sparse_path,index=False)
print('TF-IDF+SVC predictions:',len(sparse_pred))
print('Saved:',sparse_path)

In [ ]:
# CELL 6 — External performance, per-class reports, one-event uncertainty, ZIP export.
from sklearn.metrics import (accuracy_score,f1_score,precision_recall_fscore_support,
                             confusion_matrix,balanced_accuracy_score,brier_score_loss)
from scipy.stats import beta
from IPython.display import display,FileLink

assert frozen_path.is_file() and len(frozen)==600
assert full_pred.message_id.tolist()==frozen.message_id.tolist()
assert sparse_pred.message_id.tolist()==frozen.message_id.tolist()

results=[]
for subset_name,mask in [
    ('all_600',pd.Series(True,index=frozen.index)),
    ('disaster_relevant_504',frozen.disaster_relevance.eq('relevant'))]:
    subset=frozen.loc[mask]
    for model_name,pr in [('VIRCON_full_seed20266008',full_pred),('TFIDF_LinearSVC',sparse_pred)]:
        p=pr.loc[mask]
        assert len(p)==len(subset)
        y=subset.humanitarian_label_11.to_numpy()
        yp=p.pred_11.to_numpy()
        present=sorted(set(y))
        results.append({'subset':subset_name,'task':'humanitarian_11',
            'model':model_name,'n':len(y),'positives_request':int(sum(y=='requests_or_needs')),
            'accuracy':accuracy_score(y,yp),
            'macro_f1_all_11':f1_score(y,yp,labels=LABELS,average='macro',zero_division=0),
            'macro_f1_present_truth':f1_score(y,yp,labels=present,average='macro',zero_division=0),
            'macro_f1_triage':np.nan,'request_recall':np.nan,
            'request_precision':np.nan,'n_requests':np.nan})
        prec,rec,f1,support=precision_recall_fscore_support(y,yp,labels=LABELS,zero_division=0)
        pd.DataFrame({'label':LABELS,'support':support,'precision':prec,'recall':rec,'f1':f1}).to_csv(
            WORK/f'{model_name}_{subset_name}_Humanitarian_PerClass.csv',index=False)
        pd.DataFrame(confusion_matrix(y,yp,labels=LABELS),index=LABELS,columns=LABELS).to_csv(
            WORK/f'{model_name}_{subset_name}_Confusion_11.csv')

        triage_mask=mask & frozen.actionability_label.isin(TRIAGE)
        true3=frozen.loc[triage_mask,'actionability_label'].to_numpy()
        pred3=pr.loc[triage_mask,'pred_triage'].to_numpy()
        assert len(true3)>0
        precision,recall,fs,ns=precision_recall_fscore_support(true3,pred3,labels=TRIAGE,zero_division=0)
        nreq=int(ns[TRIAGE.index('request')]); tp=int(np.sum((true3=='request')&(pred3=='request')))
        if nreq:
            ci_lo=0.0 if tp==0 else float(beta.ppf(.025,tp,nreq-tp+1))
            ci_hi=1.0 if tp==nreq else float(beta.ppf(.975,tp+1,nreq-tp))
        else:ci_lo=ci_hi=np.nan
        pd.DataFrame({'label':TRIAGE,'support':ns,'precision':precision,'recall':recall,'f1':fs}).to_csv(
            WORK/f'{model_name}_{subset_name}_Actionability_PerClass.csv',index=False)
        pd.DataFrame(confusion_matrix(true3,pred3,labels=TRIAGE),index=TRIAGE,columns=TRIAGE).to_csv(
            WORK/f'{model_name}_{subset_name}_Confusion_Actionability.csv')
        entry={'subset':subset_name,'task':'human_actionability_3',
            'model':model_name,'n':len(true3),'positives_request':nreq,
            'accuracy':accuracy_score(true3,pred3),
            'macro_f1_all_11':np.nan,'macro_f1_present_truth':np.nan,
            'macro_f1_triage':f1_score(true3,pred3,labels=TRIAGE,average='macro',zero_division=0),
            'request_recall':recall[TRIAGE.index('request')],
            'request_precision':precision[TRIAGE.index('request')],
            'request_recall_exact95_lo':ci_lo,'request_recall_exact95_hi':ci_hi,
            'n_requests':nreq,
            'alerts_per_1000':1000*sum(pred3=='request')/len(true3),
            'missed_requests_per_1000':1000*sum((true3=='request')&(pred3!='request'))/len(true3)}
        if 'request_probability' in pr.columns:
            scores=pr.loc[triage_mask,'request_probability'].to_numpy(dtype=float)
            targets=(true3=='request').astype(int)
            entry['request_score_brier']=brier_score_loss(targets,scores)
            entry['request_score_fixed_0_5_recall']=(
                float(np.mean(scores[targets==1]>=0.5)) if targets.sum() else np.nan)
            entry['request_score_fixed_0_5_alerts_per_1000']=1000*float(np.mean(scores>=0.5))
        results.append(entry)

scores=pd.DataFrame(results)
scores.to_csv(WORK/'IJIES_External_Event_Model_Comparison.csv',index=False)
print(scores[['subset','task','model','n','positives_request','accuracy','macro_f1_all_11','macro_f1_triage','request_recall']].to_string(index=False))

# One-event message bootstrap for paired 11-class macro-F1 difference.
# This is NOT an event-cluster CI and is not evidence of transfer across disasters.
rng=np.random.default_rng(20261121)
paired_boot=[]
for subset_name,mask in [('all_600',np.ones(600,dtype=bool)),
                          ('disaster_relevant_504',frozen.disaster_relevance.eq('relevant').to_numpy())]:
    y=frozen.loc[mask,'humanitarian_label_11'].to_numpy()
    a=full_pred.loc[mask,'pred_11'].to_numpy()
    b=sparse_pred.loc[mask,'pred_11'].to_numpy()
    deltas=[]
    for _ in range(1000):
        ix=rng.integers(0,len(y),size=len(y))
        deltas.append(f1_score(y[ix],a[ix],labels=LABELS,average='macro',zero_division=0)
                      -f1_score(y[ix],b[ix],labels=LABELS,average='macro',zero_division=0))
    paired_boot.append({'subset':subset_name,'unit':'message_within_one_event',
        'paired_delta_full_minus_svc':float(f1_score(y,a,labels=LABELS,average='macro',zero_division=0)
                                            -f1_score(y,b,labels=LABELS,average='macro',zero_division=0)),
        'bootstrap_95_low':float(np.quantile(deltas,.025)),
        'bootstrap_95_high':float(np.quantile(deltas,.975)),
        'WARNING':'Not independent-disaster/event-level uncertainty'})
pd.DataFrame(paired_boot).to_csv(WORK/'Message_Bootstrap_Conditional_One_Event.csv',index=False)

run_meta={'test_dataset_sha256':label_hash,'full_checkpoint_sha256':ckpt_hash,
          'historical_train_tsv_sha256':train_sha,
          'full_model_variant':'full','full_model_seed':20266008,
          'historical_training_records':int(len(train)),
          'new_events':int(frozen.event_id.nunique()),
          'labels_from_new_event_used_for_training':False,
          'threshold_selection_on_new_event':False,
          'hard_override':False,
          'comparison_note':'The SVC triage is taxonomy-derived; VIRCON triage uses its direct head.',
          'limitations':['one new disaster event','very low operational request support',
             'humanitarian and actionability taxonomies are not identical',
             'no crisis-domain encoder checkpoint in this comparison',
             'no cross-event statistical inference from one event']}
(WORK/'IJIES_External_Evaluation_Manifest.json').write_text(json.dumps(run_meta,indent=2))

zip_path=WORK/'IJIES_External_Frozen_Evaluation_Results_No_Text.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(WORK.iterdir()):
        if not p.is_file() or p.name==zip_path.name:continue
        if p.name in {'IJIES_Independent_Event_FROZEN_600.csv','Five_Human_Review_Decisions.csv'}:continue
        if 'Checkpoint' in p.name or p.suffix in {'.pt','.pth'}:continue
        z.write(p,arcname=p.name)
print();print('RESULT BUNDLE:',zip_path)
print('INCLUDE IN MANUSCRIPT: precise cohort inclusion rules and checkpoint provenance.')
print('DO NOT CLAIM independent across-event confidence intervals from this 1-event dataset.')
display(FileLink(str(zip_path)))

In [ ]:

from pathlib import Path
import json, time, random, hashlib, gc, zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoModel, AutoTokenizer, AutoConfig,
    get_linear_schedule_with_warmup
)

needed = [
    'frozen', 'train', 'model', 'tokenizer',
    'clf', 'vectorizer', 'full_pred', 'sparse_pred',
    'text_input', 'LABELS', 'TRIAGE',
    'frozen_path', 'checkpoint_path', 'train_file'
]

missing = [x for x in needed if x not in globals()]

if missing:
    raise RuntimeError(
        f"Previous notebook variables missing: {missing}. "
        "Rerun previous cells 1 to 6."
    )

assert torch.cuda.is_available(), "Enable Kaggle T4 GPU."
assert len(frozen) == len(text_input) == 600
assert len(train) == 61164
assert len(LABELS) == 11
assert set(frozen.message_id) == set(full_pred.message_id)
assert set(frozen.message_id) == set(sparse_pred.message_id)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_TRAIN_SHA = (
    "5e3dd75d1dcbdba893d8d731795b51853d04708547a9bdc2bd18a0b54b82541b"
)
EXPECTED_FULL_SHA = (
    "d8a34879c1899f6994ce5dd652fdc9b5ba7a7f1db8bf00304773fc924f19e764"
)
EXPECTED_FROZEN_SHA = (
    "c58378ba9b5b7ae247fb2b44eda79459ac76b7712f3df876ee8bb8198e9ff1b5"
)

assert sha256(frozen_path) == EXPECTED_FROZEN_SHA, \
    "Frozen dataset differs from previous evaluation!"

assert sha256(train_file) == EXPECTED_TRAIN_SHA, \
    "Original training TSV does not match!"

assert sha256(checkpoint_path) == EXPECTED_FULL_SHA, \
    "Full VIRCON checkpoint does not match!"

EVAL_DIR = Path(
    "/kaggle/working/IJIES_Crisis_External_Benchmark"
)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CRISIS_DIR = EVAL_DIR / (
    "crisis_CT_M1_Complete_fixed_20266008"
)
CRISIS_DIR.mkdir(exist_ok=True)

CRISIS_ID = "crisistransformers/CT-M1-Complete"
SEED = 20266008
DEVICE = torch.device("cuda")

# Free GPU memory while keeping previous full model.
model = model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

class ThreeHeadModel(nn.Module):
    def __init__(self, encoder, n_events=60):
        super().__init__()
        self.encoder = encoder
        h = encoder.config.hidden_size

        self.dropout = nn.Dropout(0.20)
        self.humanitarian = nn.Linear(h, len(LABELS))
        self.triage = nn.Linear(h, 3)
        self.request = nn.Linear(h, 1)

        self.event_discriminator = nn.Sequential(
            nn.Linear(h, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, n_events)
        )

    def forward(self, ids, mask):
        hidden = self.encoder(
            input_ids=ids,
            attention_mask=mask
        ).last_hidden_state[:, 0]

        hidden = self.dropout(hidden)

        return (
            self.humanitarian(hidden),
            self.triage(hidden),
            self.request(hidden).squeeze(-1)
        )

def triage_from_humanitarian(label):
    if label == "requests_or_needs":
        return "request"
    if label in (
        "response_efforts",
        "donation_and_volunteering"
    ):
        return "response"
    return "other"

print("PREVIOUS RESULTS VERIFIED")
print("Frozen event messages:", len(frozen))
print("Original training records:", len(train))
print("GPU:", torch.cuda.get_device_name(0))
print("Ready for crisis-domain encoder training.")


In [ ]:

CRISIS_CKPT = CRISIS_DIR / "model_state_dict.pt"
CRISIS_MANIFEST = CRISIS_DIR / "training_manifest.json"

if CRISIS_CKPT.exists() and CRISIS_MANIFEST.exists():

    record = json.loads(CRISIS_MANIFEST.read_text())

    assert record["training_sha256"] == EXPECTED_TRAIN_SHA
    assert record["encoder"] == CRISIS_ID
    assert record["seed"] == SEED
    assert record["epochs"] == 2
    assert sha256(CRISIS_CKPT) == record["checkpoint_sha256"]
    assert (CRISIS_DIR / "config.json").exists()
    assert (CRISIS_DIR / "tokenizer").exists()

    print("Reusing previously trained crisis checkpoint.")

else:
    if CRISIS_CKPT.exists():
        raise RuntimeError(
            "Incomplete checkpoint exists. Inspect it first."
        )

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    ct_tokenizer = AutoTokenizer.from_pretrained(
        CRISIS_ID, use_fast=True
    )

    ct_encoder = AutoModel.from_pretrained(CRISIS_ID)

    ct_model = ThreeHeadModel(
        ct_encoder, n_events=60
    ).to(DEVICE)

    # Use ONLY original CrisisBench training records.
    texts = train.clean_text.astype(str).tolist()

    y = torch.tensor(
        [LABELS.index(x) for x in train.class_label.tolist()],
        dtype=torch.long
    )

    encoded = ct_tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    dataset = TensorDataset(
        encoded["input_ids"],
        encoded["attention_mask"],
        y
    )

    generator = torch.Generator().manual_seed(SEED)

    loader = DataLoader(
        dataset,
        batch_size=24,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )

    counts = torch.bincount(y, minlength=11).float()

    class_weights = (
        len(y) / (11 * counts)
    ).clamp(0.25, 8.0).to(DEVICE)

    optimizer = torch.optim.AdamW(
        ct_model.parameters(),
        lr=2e-5,
        weight_decay=0.01
    )

    steps = 2 * len(loader)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.10 * steps),
        num_training_steps=steps
    )

    scaler = torch.amp.GradScaler("cuda")

    history = []

    for epoch in range(2):

        ct_model.train()
        losses = []
        start = time.perf_counter()

        for j, (ids, mask, target) in enumerate(loader, 1):

            ids = ids.to(DEVICE, non_blocking=True)
            mask = mask.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                logits11, _, _ = ct_model(ids, mask)

                loss = F.cross_entropy(
                    logits11,
                    target,
                    weight=class_weights
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                ct_model.parameters(), 1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss.detach().item())

            if j % 400 == 0:
                print(
                    f"Epoch {epoch+1}/2 | "
                    f"Batch {j}/{len(loader)} | "
                    f"Loss {np.mean(losses):.4f}",
                    flush=True
                )

        result = {
            "epoch": epoch + 1,
            "mean_loss": float(np.mean(losses)),
            "seconds": time.perf_counter() - start
        }

        history.append(result)
        print("Completed epoch:", result, flush=True)

    # Save checkpoint for reuse.
    torch.save(
        ct_model.cpu().state_dict(),
        CRISIS_CKPT
    )

    ct_model.encoder.config.save_pretrained(CRISIS_DIR)

    ct_tokenizer.save_pretrained(
        CRISIS_DIR / "tokenizer"
    )

    pd.DataFrame(history).to_csv(
        CRISIS_DIR / "epoch_history.csv",
        index=False
    )

    record = {
        "encoder": CRISIS_ID,
        "seed": SEED,
        "epochs": 2,
        "training_sha256": EXPECTED_TRAIN_SHA,
        "n_train": 61164,
        "max_length": 128,
        "batch_size": 24,
        "learning_rate": 2e-5,
        "objective": "weighted humanitarian cross-entropy only",
        "no_external_event_training": True,
        "checkpoint_sha256": sha256(CRISIS_CKPT)
    }

    CRISIS_MANIFEST.write_text(
        json.dumps(record, indent=2)
    )

    del ct_model, ct_encoder, ct_tokenizer
    del optimizer, scheduler, scaler, loader, dataset, encoded

    gc.collect()
    torch.cuda.empty_cache()

    print("CRISIS MODEL TRAINING COMPLETED")

print("Checkpoint:", CRISIS_CKPT)
print("SHA256:", sha256(CRISIS_CKPT))


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from scipy.stats import beta

BATCH = 32
benchmark_rows = []
clean_texts = list(text_input)

assert len(clean_texts) == 600

def transformer_predict(name, net, tok, direct_triage):

    net.to(DEVICE).eval()

    batches = [
        clean_texts[i:i+BATCH]
        for i in range(0, 600, BATCH)
    ]

    def pass_once(collect=False):
        probs11 = []
        probs3 = []

        with torch.inference_mode():
            for batch in batches:

                inp = tok(
                    batch,
                    padding="max_length",
                    truncation=True,
                    max_length=128,
                    return_tensors="pt"
                )

                a, b, _ = net(
                    inp["input_ids"].to(DEVICE),
                    inp["attention_mask"].to(DEVICE)
                )

                if collect:
                    probs11.append(
                        F.softmax(a.float(), dim=1)
                        .cpu().numpy()
                    )

                    if direct_triage:
                        probs3.append(
                            F.softmax(b.float(), dim=1)
                            .cpu().numpy()
                        )

        if collect:
            return (
                np.vstack(probs11),
                np.vstack(probs3) if probs3 else None
            )

    # Warm-up runs.
    for _ in range(2):
        pass_once()

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    timings = []

    for _ in range(4):
        torch.cuda.synchronize()
        start = time.perf_counter()

        pass_once()

        torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)

    p11, p3 = pass_once(collect=True)

    pred11 = [
        LABELS[i] for i in p11.argmax(axis=1)
    ]

    if direct_triage:
        pred3 = [
            TRIAGE[i] for i in p3.argmax(axis=1)
        ]
    else:
        pred3 = [
            triage_from_humanitarian(x)
            for x in pred11
        ]

    predictions = pd.DataFrame({
        "message_id": frozen.message_id.values,
        "pred_11": pred11,
        "pred_triage": pred3
    })

    predictions.to_csv(
        EVAL_DIR / f"{name}_Predictions_No_Text.csv",
        index=False
    )

    median = float(np.median(timings))

    benchmark_rows.append({
        "model": name,
        "device": torch.cuda.get_device_name(0),
        "n_messages": 600,
        "batch_size": BATCH,
        "seconds_median": median,
        "milliseconds_per_message": 1000 * median / 600,
        "messages_per_second": 600 / median,
        "peak_gpu_allocated_mib":
            torch.cuda.max_memory_allocated() / 1024**2,
        "parameter_million":
            sum(p.numel() for p in net.parameters()) / 1e6,
        "triage_mechanism":
            "direct_head" if direct_triage
            else "taxonomy_derived",
        "timing_scope":
            "tokenization, transfer, GPU forward; excludes model loading"
    })

    net.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()

    print(
        name,
        "| predictions:", len(predictions),
        "| median seconds:", round(median, 3)
    )

    return predictions


# 1. Crisis-specific encoder.
ct_tok = AutoTokenizer.from_pretrained(
    str(CRISIS_DIR / "tokenizer"),
    local_files_only=True
)

ct_cfg = AutoConfig.from_pretrained(
    str(CRISIS_DIR),
    local_files_only=True
)

ct_model = ThreeHeadModel(
    AutoModel.from_config(ct_cfg),
    n_events=60
)

ct_model.load_state_dict(
    torch.load(
        CRISIS_CKPT,
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)

crisis_predictions = transformer_predict(
    "Crisis_CT_M1",
    ct_model,
    ct_tok,
    direct_triage=False
)

del ct_model, ct_tok
gc.collect()


# 2. Previously trained full VIRCON-ERT.
full_benchmark_predictions = transformer_predict(
    "VIRCON_full",
    model,
    tokenizer,
    direct_triage=True
)

# Verify predictions against previous notebook.
check = full_benchmark_predictions.merge(
    full_pred[[
        "message_id", "pred_11", "pred_triage"
    ]],
    on="message_id",
    suffixes=("_new", "_old"),
    validate="one_to_one"
)

assert (
    check.pred_11_new == check.pred_11_old
).all(), "Full model humanitarian predictions changed!"

assert (
    check.pred_triage_new == check.pred_triage_old
).all(), "Full model triage predictions changed!"

print("VIRCON-ERT reproducibility check: PASSED")


# 3. Previously fitted TF-IDF + LinearSVC.
def sparse_pass():
    return clf.predict(
        vectorizer.transform(clean_texts)
    )

for _ in range(2):
    sparse_pass()

times = []

for _ in range(4):
    start = time.perf_counter()
    sparse_pass()
    times.append(time.perf_counter() - start)

svc_predictions = pd.DataFrame({
    "message_id": frozen.message_id.values,
    "pred_11": sparse_pass()
})

svc_predictions["pred_triage"] = (
    svc_predictions.pred_11.map(
        triage_from_humanitarian
    )
)

svc_predictions.to_csv(
    EVAL_DIR / "TFIDF_LinearSVC_Predictions_No_Text.csv",
    index=False
)

check = svc_predictions.merge(
    sparse_pred[[
        "message_id", "pred_11", "pred_triage"
    ]],
    on="message_id",
    suffixes=("_new", "_old"),
    validate="one_to_one"
)

assert (
    check.pred_11_new == check.pred_11_old
).all(), "Sparse baseline predictions changed!"

median = float(np.median(times))

benchmark_rows.append({
    "model": "TFIDF_LinearSVC",
    "device": "CPU (same Kaggle session)",
    "n_messages": 600,
    "batch_size": 600,
    "seconds_median": median,
    "milliseconds_per_message": 1000 * median / 600,
    "messages_per_second": 600 / median,
    "peak_gpu_allocated_mib": np.nan,
    "parameter_million": np.nan,
    "triage_mechanism": "taxonomy_derived",
    "timing_scope": "CPU vectorization + prediction; excludes fitting"
})

benchmark = pd.DataFrame(benchmark_rows)

benchmark.to_csv(
    EVAL_DIR / "Same_Hardware_Inference_Benchmark.csv",
    index=False
)

print("\nSAME-SESSION BENCHMARK")
print(
    benchmark[[
        "model",
        "device",
        "milliseconds_per_message",
        "messages_per_second"
    ]].to_string(index=False)
)


In [ ]:

truth = frozen.set_index("message_id")

models = {
    "VIRCON_full": full_benchmark_predictions,
    "Crisis_CT_M1": crisis_predictions,
    "TFIDF_LinearSVC": svc_predictions
}

rows = []

for name, pred in models.items():

    p = pred.set_index("message_id").loc[truth.index]

    populations = [
        ("all_600", np.ones(600, dtype=bool)),
        (
            "relevant_504",
            truth.disaster_relevance.eq("relevant").to_numpy()
        )
    ]

    for population, mask in populations:

        y11 = truth.humanitarian_label_11.to_numpy()[mask]
        yp11 = p.pred_11.to_numpy()[mask]

        per = precision_recall_fscore_support(
            y11, yp11,
            labels=LABELS,
            zero_division=0
        )

        pd.DataFrame({
            "label": LABELS,
            "precision": per[0],
            "recall": per[1],
            "f1": per[2],
            "support": per[3]
        }).to_csv(
            EVAL_DIR / f"{name}_{population}_PerClass11.csv",
            index=False
        )

        pd.DataFrame(
            confusion_matrix(y11, yp11, labels=LABELS),
            index=LABELS,
            columns=LABELS
        ).to_csv(
            EVAL_DIR / f"{name}_{population}_Confusion11.csv"
        )

        triage_mask = (
            mask &
            truth.actionability_label.isin(TRIAGE).to_numpy()
        )

        y3 = truth.actionability_label.to_numpy()[triage_mask]
        yp3 = p.pred_triage.to_numpy()[triage_mask]

        tp = int(((y3 == "request") & (yp3 == "request")).sum())
        fp = int(((y3 != "request") & (yp3 == "request")).sum())
        fn = int(((y3 == "request") & (yp3 != "request")).sum())

        npos = tp + fn

        if npos:
            ci_low = (
                0 if tp == 0
                else float(beta.ppf(.025, tp, npos-tp+1))
            )
            ci_high = (
                1 if tp == npos
                else float(beta.ppf(.975, tp+1, npos-tp))
            )
        else:
            ci_low = ci_high = np.nan

        pd.DataFrame(
            confusion_matrix(y3, yp3, labels=TRIAGE),
            index=TRIAGE,
            columns=TRIAGE
        ).to_csv(
            EVAL_DIR / f"{name}_{population}_Confusion3.csv"
        )

        rows.append({
            "model": name,
            "population": population,
            "n_humanitarian": len(y11),
            "macro_f1_11": f1_score(
                y11, yp11,
                labels=LABELS,
                average="macro",
                zero_division=0
            ),
            "accuracy_11": accuracy_score(y11, yp11),
            "n_actionability": len(y3),
            "macro_f1_actionability": f1_score(
                y3, yp3,
                labels=TRIAGE,
                average="macro",
                zero_division=0
            ),
            "n_requests": npos,
            "request_TP": tp,
            "request_FP": fp,
            "request_FN": fn,
            "request_precision":
                tp / (tp + fp) if tp + fp else np.nan,
            "request_recall":
                tp / npos if npos else np.nan,
            "request_recall_CI_low": ci_low,
            "request_recall_CI_high": ci_high,
            "false_request_alerts_per_1000":
                1000 * fp / len(y3)
        })

results = pd.DataFrame(rows)

results.to_csv(
    EVAL_DIR / "External_Model_Comparison_3_Models.csv",
    index=False
)

print("\nTHREE-MODEL EXTERNAL COMPARISON")
print(
    results[[
        "model", "population",
        "macro_f1_11",
        "macro_f1_actionability",
        "request_precision",
        "request_recall",
        "request_FP"
    ]].to_string(index=False)
)


# Paired bootstrap: messages within ONE event.
rng = np.random.default_rng(20261121)
boot = []

for pop, mask in [
    ("all_600", np.ones(600, dtype=bool)),
    (
        "relevant_504",
        truth.disaster_relevance.eq("relevant").to_numpy()
    )
]:

    y = truth.humanitarian_label_11.to_numpy()[mask]
    a = full_benchmark_predictions.pred_11.to_numpy()[mask]
    b = crisis_predictions.pred_11.to_numpy()[mask]

    def macro_f1(y_true, y_pred):
        return f1_score(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )

    point = macro_f1(y, a) - macro_f1(y, b)

    draws = []

    for _ in range(2000):
        idx = rng.integers(0, len(y), size=len(y))
        draws.append(
            macro_f1(y[idx], a[idx]) -
            macro_f1(y[idx], b[idx])
        )

    lo, hi = np.quantile(draws, [.025, .975])

    boot.append({
        "population": pop,
        "VIRCON_minus_Crisis_macro_f1": point,
        "CI95_low": float(lo),
        "CI95_high": float(hi),
        "unit": "message within one event",
        "resamples": 2000
    })

pd.DataFrame(boot).to_csv(
    EVAL_DIR / "Paired_Bootstrap_Full_vs_Crisis.csv",
    index=False
)

# Reproducibility manifest.
report = {
    "test_dataset_SHA256": sha256(frozen_path),
    "original_train_SHA256": EXPECTED_TRAIN_SHA,
    "full_checkpoint_SHA256": EXPECTED_FULL_SHA,
    "crisis_checkpoint_SHA256": sha256(CRISIS_CKPT),
    "crisis_model": CRISIS_ID,
    "seed": SEED,
    "crisis_epochs": 2,
    "external_labels_used_to_train": False,
    "triage_VIRCON": "trained direct head",
    "triage_Crisis_and_SVC":
        "mapped from predicted humanitarian label",
    "inference_device_transformers":
        torch.cuda.get_device_name(0),
    "sparse_baseline_device": "CPU",
    "limitations": [
        "one unseen disaster",
        "rare operational request positives",
        "message bootstrap does not quantify cross-event generalization"
    ]
}

(EVAL_DIR / "External_Crisis_Benchmark_Manifest.json").write_text(
    json.dumps(report, indent=2)
)

zip_path = Path(
    "/kaggle/working/"
    "IJIES_Crisis_External_Benchmark_Results_No_Text.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for path in EVAL_DIR.glob("*"):
        if path.is_file() and path.suffix in (".csv", ".json"):
            z.write(path, arcname=path.name)

print("\nEXPERIMENT COMPLETED")
print("Results ZIP:", zip_path)
print("Saved crisis checkpoint:", CRISIS_CKPT)


In [ ]:
# No neural-network training, no model downloads, no new-event label changes.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from io import BytesIO
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support

OUT11 = Path('/kaggle/working/IJIES_Post_Benchmark_Analysis')
OUT11.mkdir(parents=True, exist_ok=True)
BENCH11 = Path('/kaggle/working/IJIES_Crisis_External_Benchmark')
LABELS11 = [
    'affected_individual', 'caution_and_advice', 'displaced_and_evacuations',
    'donation_and_volunteering', 'infrastructure_and_utilities_damage',
    'injured_or_dead_people', 'missing_and_found_people', 'not_humanitarian',
    'requests_or_needs', 'response_efforts', 'sympathy_and_support'
]
TRIAGE11 = ['other', 'request', 'response']
FROZEN_SHA11 = 'c58378ba9b5b7ae247fb2b44eda79459ac76b7712f3df876ee8bb8198e9ff1b5'

def _candidate_files11(name):
    found=[]
    for root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        if root.exists():
            found.extend(p for p in root.rglob(name) if p.is_file())
    return sorted(set(found), key=lambda p: (len(str(p)),str(p)))

def _hash11(path):
    h=hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Reuse the already-loaded frozen dataset; otherwise read its frozen CSV only.
if 'frozen' in globals() and isinstance(frozen, pd.DataFrame):
    audit11=frozen.copy()
    print('Using frozen ground truth already held in this Kaggle notebook.')
else:
    candidates11=_candidate_files11('IJIES_Independent_Event_FROZEN_600.csv')
    if not candidates11:
        raise FileNotFoundError('Frozen ground truth missing. Run the ORIGINAL Cell 1 only, or attach IJIES_Independent_Event_FROZEN_600.csv via Add Input. Do not rerun model training.')
    frozen_file11=next((p for p in candidates11 if _hash11(p)==FROZEN_SHA11), None)
    if frozen_file11 is None:
        raise ValueError('Frozen CSV does not match the prior benchmark SHA256; stop before statistical comparisons.')
    audit11=pd.read_csv(frozen_file11, dtype=str).fillna('')
    print('Loaded and SHA256-verified:',frozen_file11)

# If the runtime still has the actual frozen file path, verify it again.
if 'frozen_path' in globals() and Path(frozen_path).is_file():
    assert _hash11(Path(frozen_path))==FROZEN_SHA11, 'Frozen file checksum does not match the published comparison.'

mandatory11={'message_id','humanitarian_label_11','actionability_label','disaster_relevance'}
assert mandatory11.issubset(audit11.columns), f'Missing frozen columns: {mandatory11-set(audit11.columns)}'
for col in mandatory11:
    audit11[col]=audit11[col].astype(str).str.strip()
assert len(audit11)==600 and audit11.message_id.is_unique
assert audit11.humanitarian_label_11.isin(LABELS11).all()
assert audit11.actionability_label.isin(TRIAGE11+['uncertain']).all()
assert audit11.disaster_relevance.isin(['relevant','not_relevant','uncertain']).all()
assert int(audit11.disaster_relevance.eq('relevant').sum())==504

archives11=_candidate_files11('IJIES_Crisis_External_Benchmark_Results_No_Text.zip')
models11={
    'VIRCON_full':['VIRCON_full_External_Predictions_No_Text.csv','VIRCON_full_Predictions_No_Text.csv'],
    'Crisis_CT_M1':['Crisis_CT_M1_External_Predictions_No_Text.csv','Crisis_CT_M1_Predictions_No_Text.csv'],
    'TFIDF_LinearSVC':['TFIDF_LinearSVC_External_Predictions_No_Text.csv','TFIDF_LinearSVC_Predictions_No_Text.csv']
}
preds11={}
for model, names in models11.items():
    frame=None
    for name in names:
        direct=BENCH11/name
        hits=[direct] if direct.is_file() else _candidate_files11(name)
        if hits:
            frame=pd.read_csv(hits[0], dtype=str)
            break
    if frame is None:
        for zpath in archives11:
            with ZipFile(zpath) as z:
                found=next((n for n in z.namelist() if Path(n).name in names),None)
                if found:
                    frame=pd.read_csv(BytesIO(z.read(found)), dtype=str)
                    break
    if frame is None:
        raise FileNotFoundError(f'Missing {model} predictions. Keep prior Kaggle results, or Add Input: IJIES_Crisis_External_Benchmark_Results_No_Text.zip')
    needed={'message_id','pred_11','pred_triage'}
    assert needed.issubset(frame.columns) and frame.message_id.is_unique, model
    assert len(frame)==600 and set(frame.message_id)==set(audit11.message_id), model
    frame=frame.set_index('message_id').loc[audit11.message_id].reset_index()
    assert frame.pred_11.isin(LABELS11).all() and frame.pred_triage.isin(TRIAGE11).all(),model
    preds11[model]=frame
    print(f'Loaded {model}: {len(frame)} aligned predictions')

# Paired message-level bootstrap. Resamples the same IDs for each model (never train here).
rng11=np.random.default_rng(20261121)
boot11=[]
summary11=[]
rare11=[]
confusions11=[]
request_alerts11=[]
disagreements11=[]
for pop11, subset11 in [('all_600',np.ones(600,dtype=bool)),
                        ('relevant_504',audit11.disaster_relevance.eq('relevant').to_numpy())]:
    y11=audit11.humanitarian_label_11.to_numpy()[subset11]
    n11=int(subset11.sum())
    p11={model: df.pred_11.to_numpy()[subset11] for model,df in preds11.items()}
    for other11 in ('TFIDF_LinearSVC','Crisis_CT_M1'):
        a11=p11['VIRCON_full'];b11=p11[other11]
        delta11=float(f1_score(y11,a11,labels=LABELS11,average='macro',zero_division=0)
                      -f1_score(y11,b11,labels=LABELS11,average='macro',zero_division=0))
        draws11=np.empty(2000,dtype=float)
        for k11 in range(len(draws11)):
            sample11=rng11.integers(0,n11,size=n11)
            draws11[k11]=(f1_score(y11[sample11],a11[sample11],labels=LABELS11,average='macro',zero_division=0)
                         -f1_score(y11[sample11],b11[sample11],labels=LABELS11,average='macro',zero_division=0))
        lo11,hi11=np.quantile(draws11,[0.025,0.975])
        boot11.append({'population':pop11,'comparison':f'VIRCON_full minus {other11}',
                       'n_messages':n11,'macro_f1_difference':delta11,
                       'CI95_low':float(lo11),'CI95_high':float(hi11),
                       'CI_excludes_zero':bool(lo11>0 or hi11<0),'resamples':2000,
                       'sampling_unit':'message, conditional on one event'})
    mask3_11=subset11 & audit11.actionability_label.isin(TRIAGE11).to_numpy()
    y3_11=audit11.actionability_label.to_numpy()[mask3_11]
    for model11,df11 in preds11.items():
        prediction11=df11.pred_11.to_numpy()[subset11]
        pr11,re11,f111,s11=precision_recall_fscore_support(
            y11,prediction11,labels=LABELS11,zero_division=0)
        for lab11,precision11,recall11,f1class11,support11 in zip(LABELS11,pr11,re11,f111,s11):
            rare11.append({'population':pop11,'model':model11,'class':lab11,
                           'precision':float(precision11),'recall':float(recall11),
                           'f1':float(f1class11),'support':int(support11)})
        err11=pd.DataFrame({'truth':y11,'prediction':prediction11})
        err11=err11.loc[err11.truth.ne(err11.prediction)]
        pairs11=err11.groupby(['truth','prediction']).size().sort_values(ascending=False)
        for (truth11,pred11),count11 in pairs11.items():
            confusions11.append({'population':pop11,'model':model11,
                                 'true_class':truth11,'predicted_class':pred11,'count':int(count11)})
        p3_11=df11.pred_triage.to_numpy()[mask3_11]
        true_request11=y3_11=='request'
        request_prediction11=p3_11=='request'
        tp11=int(np.sum(true_request11 & request_prediction11))
        fp11=int(np.sum(~true_request11 & request_prediction11))
        fn11=int(np.sum(true_request11 & ~request_prediction11))
        summary11.append({'population':pop11,'model':model11,'n_11':n11,
                          'macro_f1_11':float(f1_score(y11,prediction11,labels=LABELS11,average='macro',zero_division=0)),
                          'n_actionability_evaluable':int(mask3_11.sum()),
                          'excluded_uncertain_actionability':int(subset11.sum()-mask3_11.sum()),
                          'request_TP':tp11,'request_FP':fp11,'request_FN':fn11,
                          'request_precision':tp11/(tp11+fp11) if tp11+fp11 else np.nan,
                          'request_recall':tp11/(tp11+fn11) if tp11+fn11 else np.nan})
        alert_mask11=mask3_11 & audit11.actionability_label.ne('request').to_numpy() & df11.pred_triage.eq('request').to_numpy()
        alerts11=pd.DataFrame({'message_id':audit11.message_id.to_numpy()[alert_mask11],
                'true_11':audit11.humanitarian_label_11.to_numpy()[alert_mask11],
                'pred_11':df11.pred_11.to_numpy()[alert_mask11],
                'true_actionability':audit11.actionability_label.to_numpy()[alert_mask11],
                'pred_actionability':df11.pred_triage.to_numpy()[alert_mask11]})
        alerts11.insert(0,'model',model11)
        alerts11.insert(0,'population',pop11)
        assert len(alerts11)==fp11,(model11,pop11,'false-alert count mismatch')
        request_alerts11.append(alerts11)
    full11=p11['VIRCON_full']
    for other11 in ('TFIDF_LinearSVC','Crisis_CT_M1'):
        right_alt11=(full11!=y11)&(p11[other11]==y11)
        right_full11=(full11==y11)&(p11[other11]!=y11)
        disagreements11.append({'population':pop11,'comparator':other11,
                    'VIRCON_wrong_comparator_correct':int(right_alt11.sum()),
                    'VIRCON_correct_comparator_wrong':int(right_full11.sum())})

# 'uncertain' is an explicit annotation state, not evidence that review is unfinished.
uncertain11=audit11.loc[audit11.actionability_label.eq('uncertain'),
                      ['message_id','disaster_relevance','humanitarian_label_11','actionability_label']]
uncertain11.to_csv(OUT11/'Excluded_Uncertain_Actionability_No_Text.csv',index=False)
pd.DataFrame(boot11).to_csv(OUT11/'Paired_Bootstrap_VIRCON_vs_Both_2000.csv',index=False)
pd.DataFrame(summary11).to_csv(OUT11/'Request_Alert_Audit.csv',index=False)
pd.DataFrame(rare11).to_csv(OUT11/'PerClass_Error_Analysis.csv',index=False)
pd.DataFrame(confusions11).to_csv(OUT11/'Misclassification_Pairs.csv',index=False)
pd.concat(request_alerts11,ignore_index=True).to_csv(OUT11/'False_Request_Alerts_No_Text.csv',index=False)
pd.DataFrame(disagreements11).to_csv(OUT11/'Paired_Model_Disagreements.csv',index=False)

# Optional evidence that the five cases were already human-confirmed.
confirm_files11=_candidate_files11('IJIES_Five_Label_Confirmed.csv')
confirmation11={'human_confirmation_file_found':bool(confirm_files11),
                'uncertain_actionability_count':int(len(uncertain11)),
                'annotation_changes_made_by_this_cell':0}
if confirm_files11:
    checks11=pd.read_csv(confirm_files11[0],dtype=str).fillna('')
    confirmation11['confirmation_rows']=int(len(checks11))
    if {'message_id','review_status','review_notes'}.issubset(checks11.columns):
        confirmation11['all_rows_confirmed']=bool(checks11.review_status.str.strip().str.lower().eq('confirmed').all())
        confirmation11['reviewed_ids_in_frozen']=bool(set(checks11.message_id).issubset(set(audit11.message_id)))
(OUT11/'Annotation_Audit_Notes.json').write_text(json.dumps(confirmation11,indent=2))

new_zip11=Path('/kaggle/working/IJIES_Post_Benchmark_Analysis_No_Text.zip')
with ZipFile(new_zip11,'w',compression=ZIP_DEFLATED) as z11:
    for file11 in sorted(OUT11.iterdir()):
        if file11.suffix in ('.csv','.json'):
            z11.write(file11,arcname=file11.name)
print('\nBOOTSTRAP RESULTS')
print(pd.DataFrame(boot11).to_string(index=False))
print('\nREQUEST ALERT AUDIT')
print(pd.DataFrame(summary11).to_string(index=False))
print('\nAnnotation audit:',confirmation11)
print('\nDOWNLOAD THE NEW ZIP FROM KAGGLE OUTPUT:',new_zip11)


In [ ]:

import pandas as pd
from pathlib import Path
from IPython.display import display

OUT = Path("/kaggle/working/IJIES_Post_Benchmark_Analysis")

print("=== Statistical comparison ===")
display(pd.read_csv(OUT / "Paired_Bootstrap_VIRCON_vs_Both_2000.csv"))

print("=== Request detection and false alerts ===")
display(pd.read_csv(OUT / "Request_Alert_Audit.csv"))

print("=== Model disagreements ===")
display(pd.read_csv(OUT / "Paired_Model_Disagreements.csv"))

print("=== Most frequent classification errors ===")
errors = pd.read_csv(OUT / "Misclassification_Pairs.csv")
display(errors.sort_values("count", ascending=False).head(20))

print("\nFinal ZIP:")
print("/kaggle/working/IJIES_Post_Benchmark_Analysis_No_Text.zip")
